In [16]:
!pip install -U unsloth transformers==4.56.2 trl==0.22.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.0 MB/s eta 0:00:00
  Using cached transformers-4.56.2-py3-none-any.whl.metadata (40 kB)
Using cached transformers-4.56.2-py3-none-any.whl (11.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 MB 28.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.4.6
    Uninstalling unsloth-2026.4.6:
      Successfully uninstalled unsloth-2026.4.6


In [17]:
import os

# Notebook configuration. Update these values before running the cells below.
MODEL_CONFIG_NAME = "tiny-aya-base"
CONDITION_NAME = "phase-2-the-stack-v1-condition-1-en-5k"
TRAIN_SEEDS = [42, 123, 456]
TRAIN_SEED = TRAIN_SEEDS[0]
MODEL_CONFIG = {
    "model_id": "CohereLabs/tiny-aya-base",
    "max_seq_length": 1024,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "load_in_4bit": True,
    "learning_rate": 2.0e-4,
    "per_device_train_batch_size": 8,
    "num_train_epochs": 1,
    "warmup_ratio": 0.05,
    "max_grad_norm": 1.0,
    "use_unsloth": True,
    "gradient_accumulation_steps": 1,
    "packing": True,
    "logging_steps": 10,
    "optim": "paged_adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "cosine",
    "fp16": True,
    "bf16": False,
    "eval_strategy": "no",
    "save_strategy": "steps",
    "save_steps": 500,
    "save_total_limit": 2,
    "dataloader_num_workers": 2,
    "dataloader_pin_memory": True,
    "dataloader_persistent_workers": True,
    "bias": "none",
    "gradient_checkpointing": "unsloth",
    "full_finetuning": False,
}

if not MODEL_CONFIG_NAME:
    raise ValueError("MODEL_CONFIG_NAME must be set before running.")
if not CONDITION_NAME:
    raise ValueError("CONDITION_NAME must be set before running. See options below.")

os.environ["MODEL_CONFIG_NAME"] = MODEL_CONFIG_NAME
os.environ["CONDITION_NAME"] = CONDITION_NAME
os.environ["TRAIN_SEEDS"] = ",".join(str(seed) for seed in TRAIN_SEEDS)
os.environ["TRAIN_SEED"] = str(TRAIN_SEED)

for key, value in MODEL_CONFIG.items():
    env_key = f"MODEL_{key.upper()}"
    if isinstance(value, list):
        os.environ[env_key] = ",".join(str(item) for item in value)
    else:
        os.environ[env_key] = str(value)

print(f"Configured model: {MODEL_CONFIG_NAME} -> {MODEL_CONFIG['model_id']}")
print(f"Configured condition: {CONDITION_NAME}")
print(f"Configured seeds: {os.environ['TRAIN_SEEDS']}")


Configured model: tiny-aya-base -> CohereLabs/tiny-aya-base
Configured condition: phase-2-the-stack-v1-condition-1-en-5k
Configured seeds: 42,123,456


In [18]:
%%writefile pretokenize.py
import os

from kaggle_secrets import UserSecretsClient
from unsloth import FastModel
from datasets import load_dataset


def read_required_env(name):
    value = os.environ.get(name)
    if value is None or value == "":
        raise ValueError(f"Environment variable {name} must be set before running.")
    return value


def read_bool_env(name, default):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower() == "true"


def read_int_env(name, default):
    value = os.environ.get(name)
    return int(value) if value is not None else default


def read_float_env(name, default):
    value = os.environ.get(name)
    return float(value) if value is not None else default


MODEL_CONFIG_NAME = read_required_env("MODEL_CONFIG_NAME")
# Set the condition to train. Must match a config in
# legesher/language-decoded-data.
#
# Available conditions:
#   "condition-1-en-32k"   - English code (full 32K)
#   "condition-1-en-5k"    - English code (5K subset)
#   "condition-2-zh-5k"    - Chinese keyword-swapped (5K)
#   "condition-2-es-5k"    - Spanish keyword-swapped (5K)
#   "condition-2-ur-5k"    - Urdu keyword-swapped (5K)
#   "condition-3-zh-5k"    - Chinese mixed native (5K)
CONDITION_NAME = read_required_env("CONDITION_NAME")
MODEL_CONFIG = {
    "model_id": read_required_env("MODEL_MODEL_ID"),
    "max_seq_length": read_int_env("MODEL_MAX_SEQ_LENGTH", 1024),
    "target_modules": [module.strip() for module in read_required_env("MODEL_TARGET_MODULES").split(",") if module.strip()],
    "lora_r": read_int_env("MODEL_LORA_R", 16),
    "lora_alpha": read_int_env("MODEL_LORA_ALPHA", 32),
    "lora_dropout": read_float_env("MODEL_LORA_DROPOUT", 0.0),
    "learning_rate": read_float_env("MODEL_LEARNING_RATE", 2.0e-4),
    "per_device_train_batch_size": read_int_env("MODEL_PER_DEVICE_TRAIN_BATCH_SIZE", 8),
    "num_train_epochs": read_int_env("MODEL_NUM_TRAIN_EPOCHS", 1),
    "warmup_ratio": read_float_env("MODEL_WARMUP_RATIO", 0.05),
    "max_grad_norm": read_float_env("MODEL_MAX_GRAD_NORM", 1.0),
    "use_unsloth": read_bool_env("MODEL_USE_UNSLOTH", True),
    "load_in_4bit": read_bool_env("MODEL_LOAD_IN_4BIT", True),
    "full_finetuning": read_bool_env("MODEL_FULL_FINETUNING", False),
}
TRAIN_OUT = f"/kaggle/working/data/{MODEL_CONFIG_NAME}/{CONDITION_NAME}/train_tokenized"


def main():
    os.makedirs(os.path.dirname(TRAIN_OUT), exist_ok=True)

    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")

    if not MODEL_CONFIG.get("use_unsloth", True):
        raise ValueError("This notebook currently supports only use_unsloth=true configs.")

    print("Loaded model config from notebook environment variables.")
    print(f"Model ID: {MODEL_CONFIG['model_id']}")

    # We only need the tokenizer here.
    _, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_CONFIG["model_id"],
        max_seq_length=MODEL_CONFIG["max_seq_length"],
        load_in_4bit=MODEL_CONFIG.get("load_in_4bit", True),
        full_finetuning=MODEL_CONFIG.get("full_finetuning", False),
        token=hf_token,
    )

    train_dataset = load_dataset(
        "legesher/language-decoded-data",
        CONDITION_NAME,
        split="validation",
    )

    def tokenize_fn(batch):
        return tokenizer(
            batch["code"],
            truncation=True,
            max_length=MODEL_CONFIG["max_seq_length"],
            padding=False,
        )

    # Keep only tokenized fields to reduce disk size.
    train_tokenized = train_dataset.map(
        tokenize_fn,
        batched=True,
        num_proc=8,
        remove_columns=train_dataset.column_names,
        desc="Tokenizing train dataset",
    )

    train_tokenized.save_to_disk(TRAIN_OUT)

    print(f"Saved train tokenized dataset to: {TRAIN_OUT}")
    print(train_tokenized)


if __name__ == "__main__":
    main()


Overwriting pretokenize.py


In [19]:
%%writefile train.py
import json
import os

import torch
from kaggle_secrets import UserSecretsClient
from unsloth import FastModel
from datasets import load_from_disk
from transformers import DataCollatorForLanguageModeling, set_seed
from trl import SFTTrainer, SFTConfig


def read_required_env(name):
    value = os.environ.get(name)
    if value is None or value == "":
        raise ValueError(f"Environment variable {name} must be set before running.")
    return value


def read_bool_env(name, default):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower() == "true"


def read_int_env(name, default):
    value = os.environ.get(name)
    return int(value) if value is not None else default


def read_float_env(name, default):
    value = os.environ.get(name)
    return float(value) if value is not None else default


MODEL_CONFIG_NAME = read_required_env("MODEL_CONFIG_NAME")
DEFAULT_TRAIN_SEEDS = [42, 123, 456, 789, 1024]
TRAIN_SEEDS_ENV = os.environ.get("TRAIN_SEEDS")
if TRAIN_SEEDS_ENV:
    TRAIN_SEEDS = [int(seed.strip()) for seed in TRAIN_SEEDS_ENV.split(",") if seed.strip()]
else:
    TRAIN_SEEDS = DEFAULT_TRAIN_SEEDS
TRAIN_SEED = int(os.environ.get("TRAIN_SEED", str(TRAIN_SEEDS[0])))
# Set the condition to train. Must match a config in
# legesher/language-decoded-data.
#
# Available conditions:
#   "condition-1-en-32k"   - English code (full 32K)
#   "condition-1-en-5k"    - English code (5K subset)
#   "condition-2-zh-5k"    - Chinese keyword-swapped (5K)
#   "condition-2-es-5k"    - Spanish keyword-swapped (5K)
#   "condition-2-ur-5k"    - Urdu keyword-swapped (5K)
#   "condition-3-zh-5k"    - Chinese mixed native (5K)
def is_main_process():
    return int(os.environ.get("RANK", "0")) == 0


CONDITION_NAME = read_required_env("CONDITION_NAME")
MODEL_CONFIG = {
    "model_id": read_required_env("MODEL_MODEL_ID"),
    "max_seq_length": read_int_env("MODEL_MAX_SEQ_LENGTH", 1024),
    "target_modules": [module.strip() for module in read_required_env("MODEL_TARGET_MODULES").split(",") if module.strip()],
    "lora_r": read_int_env("MODEL_LORA_R", 16),
    "lora_alpha": read_int_env("MODEL_LORA_ALPHA", 32),
    "lora_dropout": read_float_env("MODEL_LORA_DROPOUT", 0.0),
    "learning_rate": read_float_env("MODEL_LEARNING_RATE", 2.0e-4),
    "per_device_train_batch_size": read_int_env("MODEL_PER_DEVICE_TRAIN_BATCH_SIZE", 8),
    "num_train_epochs": read_int_env("MODEL_NUM_TRAIN_EPOCHS", 1),
    "warmup_ratio": read_float_env("MODEL_WARMUP_RATIO", 0.05),
    "max_grad_norm": read_float_env("MODEL_MAX_GRAD_NORM", 1.0),
    "use_unsloth": read_bool_env("MODEL_USE_UNSLOTH", True),
    "load_in_4bit": read_bool_env("MODEL_LOAD_IN_4BIT", True),
    "gradient_accumulation_steps": read_int_env("MODEL_GRADIENT_ACCUMULATION_STEPS", 1),
    "packing": read_bool_env("MODEL_PACKING", True),
    "logging_steps": read_int_env("MODEL_LOGGING_STEPS", 10),
    "optim": os.environ.get("MODEL_OPTIM", "paged_adamw_8bit"),
    "weight_decay": read_float_env("MODEL_WEIGHT_DECAY", 0.01),
    "lr_scheduler_type": os.environ.get("MODEL_LR_SCHEDULER_TYPE", "cosine"),
    "fp16": read_bool_env("MODEL_FP16", True),
    "bf16": read_bool_env("MODEL_BF16", False),
    "eval_strategy": os.environ.get("MODEL_EVAL_STRATEGY", "no"),
    "save_strategy": os.environ.get("MODEL_SAVE_STRATEGY", "steps"),
    "save_steps": read_int_env("MODEL_SAVE_STEPS", 500),
    "save_total_limit": read_int_env("MODEL_SAVE_TOTAL_LIMIT", 2),
    "dataloader_num_workers": read_int_env("MODEL_DATALOADER_NUM_WORKERS", 2),
    "dataloader_pin_memory": read_bool_env("MODEL_DATALOADER_PIN_MEMORY", True),
    "dataloader_persistent_workers": read_bool_env("MODEL_DATALOADER_PERSISTENT_WORKERS", True),
    "bias": os.environ.get("MODEL_BIAS", "none"),
    "gradient_checkpointing": os.environ.get("MODEL_GRADIENT_CHECKPOINTING", "unsloth"),
    "full_finetuning": read_bool_env("MODEL_FULL_FINETUNING", False),
}
TRAIN_PATH = f"/kaggle/working/data/{MODEL_CONFIG_NAME}/{CONDITION_NAME}/train_tokenized"
OUTPUT_NAME = f"{CONDITION_NAME}-seed{TRAIN_SEED}"
OUTPUT_DIR = f"/kaggle/working/output/{MODEL_CONFIG_NAME}/{OUTPUT_NAME}"
UPLOAD_SUBFOLDER = f"{MODEL_CONFIG_NAME}/{OUTPUT_NAME}"


def main():
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")

    if not MODEL_CONFIG.get("use_unsloth", True):
        raise ValueError("This notebook currently supports only use_unsloth=true configs.")

    # Helpful for near-OOM situations.
    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
    set_seed(TRAIN_SEED)

    if is_main_process():
        print("Loaded model config from notebook environment variables.")
        print(f"Model ID: {MODEL_CONFIG['model_id']}")
        print(f"Condition: {CONDITION_NAME}")
        print(f"Training seed: {TRAIN_SEED}")

    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_CONFIG["model_id"],
        max_seq_length=MODEL_CONFIG["max_seq_length"],
        load_in_4bit=MODEL_CONFIG.get("load_in_4bit", True),
        full_finetuning=MODEL_CONFIG.get("full_finetuning", False),
        token=hf_token,
        # Do not set device_map="auto" under DDP
    )

    model = FastModel.get_peft_model(
        model,
        r=MODEL_CONFIG["lora_r"],
        lora_alpha=MODEL_CONFIG["lora_alpha"],
        lora_dropout=MODEL_CONFIG["lora_dropout"],
        bias=MODEL_CONFIG.get("bias", "none"),
        random_state=TRAIN_SEED,
        gradient_checkpointing=MODEL_CONFIG.get("gradient_checkpointing", "unsloth"),
        target_modules=MODEL_CONFIG["target_modules"],
    )

    train_dataset = load_from_disk(TRAIN_PATH)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    if is_main_process():
        gpu_stats = torch.cuda.get_device_properties(0)
        start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
        max_memory = round(gpu_stats.total_memory / 1024**3, 3)
        print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
        print(f"{start_gpu_memory} GB of memory reserved.")
        print(train_dataset)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        data_collator=data_collator,
        args=SFTConfig(
            output_dir=OUTPUT_DIR,
            max_seq_length=MODEL_CONFIG["max_seq_length"],
            max_grad_norm=MODEL_CONFIG["max_grad_norm"],
            per_device_train_batch_size=MODEL_CONFIG["per_device_train_batch_size"],
            gradient_accumulation_steps=MODEL_CONFIG.get("gradient_accumulation_steps", 1),
            warmup_ratio=MODEL_CONFIG["warmup_ratio"],
            num_train_epochs=MODEL_CONFIG["num_train_epochs"],
            learning_rate=MODEL_CONFIG["learning_rate"],
            packing=MODEL_CONFIG.get("packing", True),
            logging_steps=MODEL_CONFIG.get("logging_steps", 10),
            optim=MODEL_CONFIG.get("optim", "paged_adamw_8bit"),
            weight_decay=MODEL_CONFIG.get("weight_decay", 0.01),
            lr_scheduler_type=MODEL_CONFIG.get("lr_scheduler_type", "cosine"),
            report_to="none",
            fp16=MODEL_CONFIG.get("fp16", True),
            bf16=MODEL_CONFIG.get("bf16", False),
            eval_strategy=MODEL_CONFIG.get("eval_strategy", "no"),
            save_strategy=MODEL_CONFIG.get("save_strategy", "steps"),
            save_steps=MODEL_CONFIG.get("save_steps", 500),
            save_total_limit=MODEL_CONFIG.get("save_total_limit", 2),
            run_name=f"{MODEL_CONFIG_NAME}-{OUTPUT_NAME}",
            ddp_find_unused_parameters=False,
            seed=TRAIN_SEED,
            data_seed=TRAIN_SEED,
            dataloader_num_workers=MODEL_CONFIG.get("dataloader_num_workers", 2),
            dataloader_pin_memory=MODEL_CONFIG.get("dataloader_pin_memory", True),
            dataloader_persistent_workers=MODEL_CONFIG.get("dataloader_persistent_workers", True),
        ),
    )

    trainer_stats = trainer.train()

    if is_main_process():
        used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
        print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
        print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
        print(f"Peak reserved memory = {used_memory} GB.")

        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

        metrics_payload = {
            "model_config_name": MODEL_CONFIG_NAME,
            "condition_name": CONDITION_NAME,
            "seed": TRAIN_SEED,
            "output_name": OUTPUT_NAME,
            "train_result": trainer_stats.metrics,
            "log_history": trainer.state.log_history,
        }
        with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w") as f:
            json.dump(metrics_payload, f, indent=2)

        print(f"Saved model + metrics to: {OUTPUT_DIR}")

        from huggingface_hub import HfApi

        api = HfApi()
        """
        api.upload_folder(
            folder_path=OUTPUT_DIR,
            path_in_repo=UPLOAD_SUBFOLDER,
            repo_id="legesher/language-decoded-lora",
            repo_type="model",
            token=hf_token,
            # Only upload adapter weights + per-seed metrics, skip tokenizer files
            # (tokenizer is already in base model) and checkpoint folders.
            allow_patterns=["adapter_*", "training_metrics.json"],
        )
        print(f"Uploaded adapter + metrics to legesher/language-decoded-lora/{UPLOAD_SUBFOLDER}")
    """
if __name__ == "__main__":
    main()


Overwriting train.py


In [20]:
!python pretokenize.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-22 16:36:15.642159: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776875775.670883    1112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776875775.679814    1112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776875775.701410    1112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776875775.701441    1112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

In [21]:
%%bash
TRAIN_SEEDS="${TRAIN_SEEDS:-}"

if [ -z "$TRAIN_SEEDS" ]; then
  echo "TRAIN_SEEDS must be set by the notebook config cell" >&2
  exit 1
fi

IFS=',' read -r -a SEEDS <<< "$TRAIN_SEEDS"

for seed in "${SEEDS[@]}"; do
  TRAIN_SEED="$seed" torchrun --standalone --nnodes=1 --nproc_per_node=2 train.py
done

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
Loaded model config from notebook environment variables.
Model ID: CohereLabs/tiny-aya-base
Condition: phase-2-the-stack-v1-condition-1-en-5k
Training seed: 42
==((====))==  Unsloth 2026.4.7: Fast Cohere2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2026.4.7: Fast Cohere2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs =

W0422 16:37:24.048000 1291 torch/distributed/run.py:852] 
W0422 16:37:24.048000 1291 torch/distributed/run.py:852] *****************************************
W0422 16:37:24.048000 1291 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0422 16:37:24.048000 1291 torch/distributed/run.py:852] *****************************************
[W422 16:37:24.998602548 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-04-22 16:37:29.038827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-22 16:37:29.038822: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory f